# 02 - Data Preprocessing


- Clean the `TotalCharges` column
- Remove unnecessary columns
- Encode the target variable
- Split the data into training, validation, and test sets
- Encode categorical features
- Scale numerical features
- Save the processed datasets


### Import libraries

In [46]:
import os
import random
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Set the random seed

In [47]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

### Load the dataset

In [48]:
df = pd.read_csv("../DataSet/Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [49]:
print(f"Dataset shape: {df.shape}")

Dataset shape: (7032, 21)


### Create a copy

In [50]:
data = df.copy()

### Convert TotalCharges to numeric

In [51]:
data["TotalCharges"] = pd.to_numeric(
    data["TotalCharges"],
    errors="coerce"
)

In [52]:
print(data["TotalCharges"].dtype)
print("Missing TotalCharges:", data["TotalCharges"].isnull().sum())

float64
Missing TotalCharges: 0


### Check rows with missing TotalCharges

In [53]:
data[data["TotalCharges"].isnull()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


In [54]:
data.loc[
    data["TotalCharges"].isnull(),
    ["tenure", "MonthlyCharges", "TotalCharges"]
]

,tenure,MonthlyCharges,TotalCharges


### Remove customerID

In [55]:
data = data.drop(columns=["customerID"])
data.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

### Encode the target variable

In [56]:
data["Churn"] = data["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [57]:
data["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [58]:
print("Missing target values:", data["Churn"].isnull().sum())

Missing target values: 0


In [59]:
print("Dataset shape:", data.shape)
print("Missing values:", data.isnull().sum().sum())
print("Duplicate rows:", data.duplicated().sum())

Dataset shape: (7032, 20)
Missing values: 0
Duplicate rows: 22


### Create X and y

In [60]:
X = data.drop(columns=["Churn"])
y = data["Churn"]

In [61]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (7032, 19)
Target shape: (7032,)


### Identify numerical and categorical columns

In [62]:
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

C:\Users\htoot\AppData\Local\Temp\ipykernel_1784\1913175022.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [63]:
print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


### Move SeniorCitizen to categorical columns

In [64]:
numerical_columns.remove("SeniorCitizen")
categorical_columns.append("SeniorCitizen")

In [65]:
print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']


### Create the training and temporary sets

In [66]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

### Create validation and test sets

In [67]:
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

### Check the split sizes

In [68]:
print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))
print("Test samples:", len(X_test))

Training samples: 4922
Validation samples: 1055
Test samples: 1055


### Check the target distribution

In [69]:
print("Full dataset:")
print(y.value_counts(normalize=True).round(3))

print("\nTraining set:")
print(y_train.value_counts(normalize=True).round(3))

print("\nValidation set:")
print(y_validation.value_counts(normalize=True).round(3))

print("\nTest set:")
print(y_test.value_counts(normalize=True).round(3))

Full dataset:
Churn
0    0.734
1    0.266
Name: proportion, dtype: float64

Training set:
Churn
0    0.734
1    0.266
Name: proportion, dtype: float64

Validation set:
Churn
0    0.734
1    0.266
Name: proportion, dtype: float64

Test set:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


## Create the Preprocessing Pipeline

### Numerical preprocessing

In [70]:
numerical_transformer = StandardScaler()

### Categorical preprocessing

In [71]:
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

### Combine the preprocessing steps

In [72]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_transformer,
            numerical_columns
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_columns
        )
    ]
)

## Fit Only on the Training Data

### Fit and transform the training set

In [73]:
X_train_processed = preprocessor.fit_transform(X_train)

### Transform validation and test sets

In [74]:
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

### Check the processed shapes

In [75]:
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_validation_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (4922, 46)
Processed validation shape: (1055, 46)
Processed test shape: (1055, 46)


## Add Feature Names

### Get the transformed feature names

In [76]:
feature_names = preprocessor.get_feature_names_out()

In [77]:
feature_names = [
    name.replace("numerical__", "").replace("categorical__", "")
    for name in feature_names
]

In [78]:
print("Number of processed features:", len(feature_names))
print(feature_names)

Number of processed features: 46
['tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_No', 'PaperlessBilling_Yes', 'PaymentMethod_Bank transfer (automatic)', 'Payme

### Convert processed arrays to DataFrames

In [79]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_validation_processed = pd.DataFrame(
    X_validation_processed,
    columns=feature_names,
    index=X_validation.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [80]:
X_train_processed.head()

,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,...,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen_0,SeniorCitizen_1
4491,-0.833469,0.444749,-0.607066,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1928,-0.508058,-1.492135,-0.823672,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
4660,-1.240233,-0.120451,-0.950975,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
5672,0.061411,-0.021293,-0.081500,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3604,-0.833469,1.166949,-0.495086,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0


## Combine Features and Targets

### Reset the indexes

In [81]:
X_train_processed = X_train_processed.reset_index(drop=True)
X_validation_processed = X_validation_processed.reset_index(drop=True)
X_test_processed = X_test_processed.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_validation = y_validation.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

### Add the target column

In [82]:
train_processed = X_train_processed.copy()
train_processed["Churn"] = y_train

validation_processed = X_validation_processed.copy()
validation_processed["Churn"] = y_validation

test_processed = X_test_processed.copy()
test_processed["Churn"] = y_test

In [83]:
train_processed.head()

,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,...,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen_0,SeniorCitizen_1,Churn
0,-0.833469,0.444749,-0.607066,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1
1,-0.508058,-1.492135,-0.823672,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0
2,-1.240233,-0.120451,-0.950975,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0
3,0.061411,-0.021293,-0.081500,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
4,-0.833469,1.166949,-0.495086,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1


## Validate the Processed Data

### Check missing values

In [84]:
print(
    "Training missing values:",
    train_processed.isnull().sum().sum()
)

print(
    "Validation missing values:",
    validation_processed.isnull().sum().sum()
)

print(
    "Test missing values:",
    test_processed.isnull().sum().sum()
)

Training missing values: 0
Validation missing values: 0
Test missing values: 0


### Check data types

In [85]:
train_processed.dtypes.value_counts()

float64    46
int64       1
Name: count, dtype: int64

### Check the final shapes

In [86]:
print("Training shape:", train_processed.shape)
print("Validation shape:", validation_processed.shape)
print("Test shape:", test_processed.shape)

Training shape: (4922, 47)
Validation shape: (1055, 47)
Test shape: (1055, 47)


### Create output folders

In [87]:
os.makedirs("../DataSet/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

### Save the datasets

In [88]:
train_processed.to_csv(
    "../DataSet/processed/telco_churn_train.csv",
    index=False
)

validation_processed.to_csv(
    "../DataSet/processed/telco_churn_validation.csv",
    index=False
)

test_processed.to_csv(
    "../DataSet/processed/telco_churn_test.csv",
    index=False
)

### Save the preprocessing pipeline

In [89]:
joblib.dump(
    preprocessor,
    "../models/preprocessor.joblib"
)

['../models/preprocessor.joblib']

In [90]:
print("Files saved successfully:")

print("../DataSet/processed/telco_churn_train.csv")
print("../DataSet/processed/telco_churn_validation.csv")
print("../DataSet/processed/telco_churn_test.csv")
print("../models/preprocessor.joblib")

Files saved successfully:
../DataSet/processed/telco_churn_train.csv
../DataSet/processed/telco_churn_validation.csv
../DataSet/processed/telco_churn_test.csv
../models/preprocessor.joblib
